<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_2_Data_Science_Core/Month_05_Data_Cleaning_and_Preprocessing/Week_1_Handling_Missing_Data/Day_05_Advanced_Imputation_KNN_and_Iterative.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/user-attachments/assets/2a4adf95-3d28-41e2-90d0-d06b2e9c8fa3" alt="Learn Data Science with Emmanuel Odenyire" width="180"/>

# Day 5: Advanced Imputation — KNN and Iterative Imputer

## Phase 2 | Month 5 | Week 1 | Day 5

**🎯 Goal:** Use KNNImputer and IterativeImputer for high-accuracy missing value replacement.
**⏱️ Study Session:** ~2 hours

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N = 500
# Simulate Nairobi property valuation dataset with correlated features
floor_m2    = np.random.normal(120, 45, N).clip(30, 400)
bedrooms    = (floor_m2 / 30 + np.random.normal(0, 0.5, N)).clip(1, 8).round(0)
age_years   = np.random.uniform(1, 40, N).round(1)
price_m_kes = (2.5 + 0.06*floor_m2 + 0.7*bedrooms - 0.03*age_years
               + np.random.normal(0, 1.5, N))

df_full = pd.DataFrame({'floor_m2':floor_m2,'bedrooms':bedrooms,
                        'age_years':age_years,'price_m_kes':price_m_kes})

# Introduce 20% missing at random
df_missing = df_full.copy()
for col in ['floor_m2', 'bedrooms', 'age_years']:
    mask = np.random.rand(N) < 0.20
    df_missing.loc[mask, col] = np.nan

print(f'Missing values:\n{df_missing.isnull().sum()}')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
N = 500
floor_m2 = np.random.normal(120,45,N).clip(30,400)
bedrooms = (floor_m2/30 + np.random.normal(0,0.5,N)).clip(1,8).round(0)
age_years = np.random.uniform(1,40,N).round(1)
price_m_kes = 2.5+0.06*floor_m2+0.7*bedrooms-0.03*age_years+np.random.normal(0,1.5,N)
df_full = pd.DataFrame({'floor_m2':floor_m2,'bedrooms':bedrooms,'age_years':age_years,'price_m_kes':price_m_kes})
df_missing = df_full.copy()
for col in ['floor_m2','bedrooms','age_years']:
    df_missing.loc[np.random.rand(N)<0.20, col] = np.nan

features = ['floor_m2','bedrooms','age_years']

imputers = {
    'Mean':     SimpleImputer(strategy='mean'),
    'KNN(k=5)': KNNImputer(n_neighbors=5),
    'Iterative':IterativeImputer(max_iter=10, random_state=42),
}

for name, imp in imputers.items():
    df_imp = df_missing.copy()
    df_imp[features] = imp.fit_transform(df_missing[features])
    
    # Evaluate only on originally-missing rows
    for col in features:
        mask = df_missing[col].isnull()
        if mask.sum() > 0:
            mae = mean_absolute_error(df_full.loc[mask, col], df_imp.loc[mask, col])
            print(f'{name:12s}  {col:12s}  MAE={mae:.3f}')

## When to Use Each Imputer

| Method | Best When | Caution |
|--------|-----------|--------|
| **Mean/Median** | Simple, fast baseline | Reduces variance; ignores feature correlations |
| **KNN** | Features are correlated; medium dataset (< 100k rows) | Slow on large datasets |
| **Iterative (MICE)** | High accuracy needed; features strongly correlated | Slower; may overfit on small samples |

In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
import warnings
warnings.filterwarnings('ignore')
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
N = 500
floor_m2 = np.random.normal(120,45,N).clip(30,400)
bedrooms = (floor_m2/30 + np.random.normal(0,0.5,N)).clip(1,8).round(0)
age_years = np.random.uniform(1,40,N).round(1)
df_missing = pd.DataFrame({'floor_m2':floor_m2,'bedrooms':bedrooms,'age_years':age_years})
for col in ['floor_m2','bedrooms','age_years']:
    df_missing.loc[np.random.rand(N)<0.20, col] = np.nan

# Best practice: put imputer in a pipeline
pipeline = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler',  StandardScaler()),
])
X_transformed = pipeline.fit_transform(df_missing)
print('Pipeline output shape:', X_transformed.shape)
print('No NaNs remaining:', np.isnan(X_transformed).sum())

## 💪 Exercises

### Exercise 1
Compare KNNImputer with k=3, k=5, k=10 on the property dataset. Which k minimises MAE on `floor_m2`?

### Exercise 2
Build a full `Pipeline` for the KDHS dataset: KNNImputer → StandardScaler → fit on 80% train data, then transform 20% test data. Never fit imputer on test data!

In [ ]:
# Your code here


## 📋 Summary

- ✓ **KNNImputer**: uses nearest neighbours — good when features correlate
- ✓ **IterativeImputer**: models each feature as function of others (MICE-style)
- ✓ Always fit imputer **on training data only** — use Pipeline
- ✓ Compare imputers using MAE on held-out rows to pick the best strategy

## 🚀 Next Steps
**Day 6: Handling Duplicates and Outliers**